# DockerMini

## Objetivo

El proyecto implementa un compilador didactico que lee especificaciones DockerMini organizadas en bloques de servicios y genera un archivo `docker-compose.yml` valido. El compilador incluye analisis lexico, sintactico y semantico, y traduce las directivas declaradas a las claves equivalentes de Docker Compose.

El sistema tambien demuestra dos estrategias de generacion YAML:

1. Sugerir el YAML mediante Ollama y el modelo local `qwen2.5:0.5b`.
2. Usar un generador determinista como respaldo si Ollama no esta disponible o la salida no pasa la validacion.

## Alcance

- Declaracion de multiples servicios mediante `servicio <id> { ... }`.
- Directivas internas en cualquier orden.
- Imagen base mediante `imagen`.
- Mapeo de un puerto mediante `puerto`.
- Volumenes o bind mounts mediante `volumen`.
- Comando de inicio mediante `comando`.
- Variables de entorno mediante el sub-bloque `entorno`.
- Dependencias entre servicios mediante `depende_de`.
- Comentarios de linea iniciados con `#`.
- Cadenas con comillas simples o dobles.
- Deteccion de errores lexicos, sintacticos y semanticos.
- Escritura del archivo `<entrada>.dockermini.yml` solo cuando la compilacion termina sin errores.

## Ejemplo

```text
# Backend con base de datos y persistencia
servicio db {
    imagen: "postgres:15-alpine"
    puerto: 5432
    volumen: "datos_db:/var/lib/postgresql/data"
    entorno: {
        POSTGRES_USER: "admin"
        POSTGRES_PASSWORD: "secretpassword"
        POSTGRES_DB: "sistema_db"
    }
}

servicio api {
    imagen: "node:18-alpine"
    puerto: 3000
    volumen: ".:/app"
    comando: "npm run start:dev"
    depende_de: [db]
}
```

Entrada:

```text
servicio db {
    imagen: "postgres:15-alpine"
    puerto: 5432
}

servicio api {
    imagen: "node:18-alpine"
    puerto: 3000
    depende_de: [db]
}
```

Salida:

```yaml
version: '3.8'
services:
  db:
    image: postgres:15-alpine
    ports:
      - "5432:5432"
  api:
    image: node:18-alpine
    ports:
      - "3000:3000"
    depends_on:
      - db
```

## Análisis léxico

| Elemento | Token |
|---|---|
| `servicio` | `SERVICIO` |
| `imagen` | `IMAGEN` |
| `puerto` | `PUERTO` |
| `volumen` | `VOLUMEN` |
| `comando` | `COMANDO` |
| `entorno` | `ENTORNO` |
| `depende_de` | `DEPENDE_DE` |
| `{` y `}` | `LLAVE_A`, `LLAVE_C` |
| `[` y `]` | `CORCHETE_A`, `CORCHETE_C` |
| `:` | `DOS_PUNTOS` |
| `,` | `COMA` |
| Numero entero | `NUMERO` |
| Cadena entre comillas | `TEXTO` |
| Nombre o ruta | `IDENTIFICADOR` |

Las expresiones regulares:

- Admiten numeros enteros no negativos mediante `\d+`.
- Admiten cadenas sin saltos de linea entre comillas dobles o simples.
- Admiten identificadores que comienzan con una letra, `_`, `.` o `/`, seguidos por letras, numeros, `_`, `-`, `.`, `/` o `:` no esta incluido.
- Descartan espacios, tabulaciones y comentarios de linea.
- Cuentan correctamente finales de linea Windows (`\r\n`) y Unix (`\n`).
- Registra caracteres ilegales sin detener el analisis, para informar todos los errores lexicos encontrados.

Los caracteres reservados o no definidos fuera de una cadena producen un error lexico. Por ejemplo, `@`, `$` y `%`.

## Análisis sintáctico

```text
programa             ::= lista_servicios
lista_servicios      ::= lista_servicios servicio | servicio
servicio             ::= SERVICIO IDENTIFICADOR LLAVE_A declaraciones LLAVE_C
declaraciones        ::= declaraciones declaracion | declaracion
declaracion          ::= IMAGEN DOS_PUNTOS valor_texto
                       | PUERTO DOS_PUNTOS NUMERO
                       | VOLUMEN DOS_PUNTOS valor_ruta
                       | COMANDO DOS_PUNTOS TEXTO
                       | ENTORNO DOS_PUNTOS LLAVE_A lista_variables LLAVE_C
                       | DEPENDE_DE DOS_PUNTOS CORCHETE_A lista_ids CORCHETE_C
valor_texto          ::= TEXTO | IDENTIFICADOR
valor_ruta           ::= TEXTO | IDENTIFICADOR
lista_variables      ::= lista_variables asignacion_var | asignacion_var
asignacion_var       ::= IDENTIFICADOR DOS_PUNTOS (valor_texto | NUMERO)
lista_ids            ::= lista_ids COMA IDENTIFICADOR | IDENTIFICADOR
```

La lista de declaraciones es recursiva, por lo que el orden de `imagen`, `puerto`, `volumen`, `comando`, `entorno` y `depende_de` puede cambiar. El parser conserva las directivas en un diccionario y luego las reordena para la salida.

## Análisis semántico

No se limita a validar la forma gramatical. Las reglas semanticas implementadas son:

- Todo servicio debe declarar `imagen`.
- Una directiva no puede repetirse dentro del mismo servicio.
- Una variable de entorno no puede repetirse dentro de su bloque.
- Un servicio no puede estar definido mas de una vez.
- Cada servicio mencionado en `depende_de` debe existir.
- Un servicio no puede depender de si mismo.

Las dependencias solo se validan si no hubo errores lexicos ni sintacticos. En `main.py`, los errores se comunican con esta prioridad:

```text
error lexico > error sintactico > error semantico
```

Si existe mas de una clase de error, solo se reporta la de mayor prioridad.

## Representacion intermedia y salida

El parser produce una estructura:

```text
(nombre_del_servicio, {directiva: valor})
```

Por ejemplo:

```text
("api", {
    "imagen": "node:18-alpine",
    "puerto": 3000,
    "volumen": ".:/app",
    "comando": "npm run start:dev",
    "depende_de": ["db"]
})
```

Luego [traductor/ag_generar_servicio.py](../traductor/ag_generar_servicio.py) transforma esta estructura en el bloque Compose correspondiente:

| DockerMini | Docker Compose |
|---|---|
| `imagen: "..."` | `image: ...` |
| `puerto: 3000` | `ports: ["3000:3000"]` |
| `volumen: "..."` | `volumes: ["..."]` |
| `comando: "..."` | `command: ...` |
| `entorno: { K: "V" }` | `environment: { K: V }` |
| `depende_de: [db, redis]` | `depends_on: [db, redis]` |

El archivo [traductor/main.py](../traductor/main.py) construye el documento completo con:

```yaml
version: '3.8'
services:
```

## Conclusiones

DockerMini demuestra la separacion entre las fases de un compilador: reconocimientos lexicos, construccion de la estructura sintactica, comprobacion semantica y generacion de codigo destino.

La gramatica permite una sintaxis declarativa flexible sin depender del orden de las directivas. La fase semantica impide configuraciones ambiguas, como directivas repetidas o dependencias inexistentes.

El uso de Ollama se trata como una optimizacion de la generacion: su salida nunca se acepta automaticamente. Primero se valida como YAML y se compara exactamente con los datos semanticos. El generador determinista garantiza que el compilador siga produciendo resultados correctos aunque el modelo o el servicio local no este disponible.


In [7]:
%%file traductor/lexer_docker.py
import ply.lex as lex

tokens = (
    'SERVICIO',
    'IMAGEN',
    'PUERTO',
    'VOLUMEN',
    'COMANDO',
    'ENTORNO',
    'DEPENDE_DE',
    'LLAVE_A',
    'LLAVE_C',
    'CORCHETE_A',
    'CORCHETE_C',
    'DOS_PUNTOS',
    'COMA',
    'NUMERO',
    'TEXTO',
    'IDENTIFICADOR',
)

reservadas = {
    'servicio': 'SERVICIO',
    'imagen': 'IMAGEN',
    'puerto': 'PUERTO',
    'volumen': 'VOLUMEN',
    'comando': 'COMANDO',
    'entorno': 'ENTORNO',
    'depende_de': 'DEPENDE_DE',
}

t_LLAVE_A     = r'\{'
t_LLAVE_C     = r'\}'
t_CORCHETE_A  = r'\['
t_CORCHETE_C  = r'\]'
t_DOS_PUNTOS  = r'\:'
t_COMA        = r'\,'

# Comentarios de línea: se descartan sin producir token
def t_COMENTARIO(t):
    r'\#.*'
    pass

# Cadenas entre comillas dobles o simples. Sin saltos de línea: una comilla
# sin cerrar queda como carácter ilegal (error léxico) en la línea correcta.
def t_TEXTO(t):
    r'(\"[^\"\n]*\"|\'[^\'\n]*\')'
    t.value = t.value[1:-1]
    return t

def t_NUMERO(t):
    r'\d+'
    t.value = int(t.value)
    return t

# Sin ':' dentro del identificador, para no absorber los dos puntos de
# 'imagen:', 'puerto:', etc. Las palabras reservadas se resuelven por diccionario.
def t_IDENTIFICADOR(t):
    r'[a-zA-Z_\.\/][a-zA-Z0-9_\-\.\/]*'
    t.type = reservadas.get(t.value, 'IDENTIFICADOR')
    return t

t_ignore = ' \t'

def t_newline(t):
    r'(\r?\n)+'
    t.lexer.lineno += t.value.count('\n')

# Error léxico: se registra (carácter, línea) y se sigue para reportar todos
def t_error(t):
    t.lexer.errores_lexicos.append((t.value[0], t.lexer.lineno))
    t.lexer.skip(1)

lexer = lex.lex()
lexer.errores_lexicos = []

def reiniciar_lexer():
    """Deja el lexer limpio para una nueva compilación."""
    lexer.errores_lexicos = []
    lexer.lineno = 1

Overwriting traductor/lexer_docker.py


In [8]:
%%file traductor/parser_docker.py
import ply.yacc as yacc
from lexer_docker import tokens, lexer, reiniciar_lexer

# Orden canónico de las directivas en la salida, sin importar cómo se escribieron
ORDEN_DIRECTIVAS = ['imagen', 'puerto', 'volumen', 'comando', 'entorno', 'depende_de']

# Resultado del análisis (se vacían en cada llamada a analizar())
servicios = []             # [(nombre, {directiva: valor})]
errores_sintacticos = []   # [mensaje]
errores_semanticos = []    # [mensaje]


# ---------------------------------------------------------------- gramática

def p_programa(p):
    """
    programa : lista_servicios
    """
    pass

def p_lista_servicios(p):
    """
    lista_servicios : lista_servicios servicio
                    | servicio
    """
    pass

def p_servicio(p):
    'servicio : SERVICIO IDENTIFICADOR LLAVE_A declaraciones LLAVE_C'
    nombre = p[2]
    linea = p.lineno(2)

    datos = {}
    for clave, valor, lin in p[4]:
        if clave in datos:
            errores_semanticos.append(
                f"Línea {lin}: la directiva '{clave}' está repetida en el servicio '{nombre}'")
        else:
            datos[clave] = valor

    if 'imagen' not in datos:
        errores_semanticos.append(
            f"Línea {linea}: el servicio '{nombre}' no declara 'imagen'")

    if any(n == nombre for n, _ in servicios):
        errores_semanticos.append(
            f"Línea {linea}: el servicio '{nombre}' está definido más de una vez")

    ordenados = {k: datos[k] for k in ORDEN_DIRECTIVAS if k in datos}
    servicios.append((nombre, ordenados))

def p_declaraciones_recursiva(p):
    'declaraciones : declaraciones declaracion'
    p[0] = p[1] + [p[2]]

def p_declaraciones_base(p):
    'declaraciones : declaracion'
    p[0] = [p[1]]

# Cada declaración devuelve (clave, valor, línea)
def p_declaracion_imagen(p):
    'declaracion : IMAGEN DOS_PUNTOS valor_texto'
    p[0] = ('imagen', p[3], p.lineno(1))

def p_declaracion_puerto(p):
    'declaracion : PUERTO DOS_PUNTOS NUMERO'
    p[0] = ('puerto', p[3], p.lineno(1))

def p_declaracion_volumen(p):
    'declaracion : VOLUMEN DOS_PUNTOS valor_ruta'
    p[0] = ('volumen', p[3], p.lineno(1))

def p_declaracion_comando(p):
    'declaracion : COMANDO DOS_PUNTOS TEXTO'
    p[0] = ('comando', p[3], p.lineno(1))

def p_declaracion_entorno(p):
    'declaracion : ENTORNO DOS_PUNTOS LLAVE_A lista_variables LLAVE_C'
    p[0] = ('entorno', p[4], p.lineno(1))

def p_declaracion_depende_de(p):
    'declaracion : DEPENDE_DE DOS_PUNTOS CORCHETE_A lista_ids CORCHETE_C'
    p[0] = ('depende_de', p[4], p.lineno(1))

def p_valor_texto(p):
    """
    valor_texto : TEXTO
                | IDENTIFICADOR
    """
    p[0] = p[1]

def p_valor_ruta(p):
    """
    valor_ruta : TEXTO
               | IDENTIFICADOR
    """
    p[0] = p[1]

def p_lista_variables_recursiva(p):
    'lista_variables : lista_variables asignacion_var'
    entorno = p[1]
    clave, valor, linea = p[2]
    if clave in entorno:
        errores_semanticos.append(
            f"Línea {linea}: la variable de entorno '{clave}' está repetida")
    else:
        entorno[clave] = valor
    p[0] = entorno

def p_lista_variables_base(p):
    'lista_variables : asignacion_var'
    clave, valor, _ = p[1]
    p[0] = {clave: valor}

def p_asignacion_var(p):
    """
    asignacion_var : IDENTIFICADOR DOS_PUNTOS valor_texto
                   | IDENTIFICADOR DOS_PUNTOS NUMERO
    """
    p[0] = (p[1], str(p[3]), p.lineno(1))

def p_lista_ids_recursiva(p):
    'lista_ids : lista_ids COMA IDENTIFICADOR'
    p[0] = p[1] + [p[3]]

def p_lista_ids_base(p):
    'lista_ids : IDENTIFICADOR'
    p[0] = [p[1]]

def p_error(p):
    # PLY puede llamar varias veces a p_error; se informa solo el primer error
    if errores_sintacticos:
        return
    if p is None:
        errores_sintacticos.append(
            "Fin de archivo inesperado (¿falta cerrar una llave o un corchete?)")
    else:
        errores_sintacticos.append(
            f"Línea {p.lineno}: token inesperado {p.type} ({p.value!r})")

parser = yacc.yacc(write_tables=False, debug=False)


# ------------------------------------------------------------ interfaz pública

def _validar_dependencias():
    nombres = {n for n, _ in servicios}
    for nombre, datos in servicios:
        for dep in datos.get('depende_de', []):
            if dep == nombre:
                errores_semanticos.append(
                    f"El servicio '{nombre}' no puede depender de sí mismo")
            elif dep not in nombres:
                errores_semanticos.append(
                    f"El servicio '{nombre}' depende de '{dep}', que no está definido")

def analizar(texto):
    """
    Fase de análisis completa (léxico + sintáctico + semántico).
    No genera YAML: eso lo hace main.py solo si no hubo errores.
    Prioridad de errores: léxico > sintáctico > semántico.
    """
    servicios.clear()
    errores_sintacticos.clear()
    errores_semanticos.clear()
    reiniciar_lexer()

    parser.parse(texto, lexer=lexer)

    if not lexer.errores_lexicos and not errores_sintacticos:
        _validar_dependencias()

    return {
        'servicios': list(servicios),
        'lexicos': list(lexer.errores_lexicos),
        'sintacticos': list(errores_sintacticos),
        'semanticos': list(errores_semanticos),
    }

Overwriting traductor/parser_docker.py


In [9]:
%%file traductor/ag_generar_servicio.py
import json
import os
import sys

import ollama
import yaml

MODELO = 'qwen2.5:0.5b'
MAX_INTENTOS = 2
# DOCKERMINI_DEBUG=1 imprime (stderr) lo que devuelve el modelo en cada intento
DEBUG = os.environ.get('DOCKERMINI_DEBUG') == '1'

SYSTEM_PROMPT = (
    "Eres el backend generador de código YAML de un compilador de Docker Compose.\n"
    "Recibirás un objeto JSON con el nombre de un servicio y sus atributos en español "
    "(imagen, puerto, volumen, comando, entorno, depende_de).\n"
    "Traduce únicamente las claves que existan en el JSON siguiendo estas reglas:\n"
    "- Nombre del servicio con 2 espacios de sangría.\n"
    "- 'imagen' -> '    image: <valor>'\n"
    "- 'puerto' -> '    ports:\\n      - \"<puerto>:<puerto>\"'\n"
    "- 'volumen' -> '    volumes:\\n      - \"<volumen>\"'\n"
    "- 'comando' -> '    command: <valor>'\n"
    "- 'entorno' -> '    environment:' seguido de cada clave: valor con 6 espacios.\n"
    "- 'depende_de' -> '    depends_on:' seguido de cada item '- <id>' con 6 espacios.\n"
    "REGLA CRÍTICA: usa exactamente las claves y los valores del JSON. NO agregues ninguna clave "
    "que no esté en el JSON (por ejemplo, no inventes 'environment' si el JSON no tiene 'entorno'), "
    "y no copies valores de los ejemplos.\n"
    "Devuelve SOLO el YAML resultante, sin bloques ```, sin 'version:' y sin 'services:'."
)

# Ejemplo 1: servicio mínimo (enseña a omitir claves no declaradas)
EJ1_IN = json.dumps({"servicio": "web",
                     "atributos": {"imagen": "nginx", "puerto": 80}}, ensure_ascii=False)
EJ1_OUT = (
    "  web:\n"
    "    image: nginx\n"
    "    ports:\n"
    "      - \"80:80\""
)

# Ejemplo 2: servicio completo (todos los mapeos y sangrías de 2, 4 y 6 espacios)
EJ2_IN = json.dumps({
    "servicio": "backend",
    "atributos": {
        "imagen": "node:18",
        "puerto": 4000,
        "volumen": ".:/app",
        "comando": "npm start",
        "entorno": {"NODE_ENV": "dev", "DB_HOST": "db"},
        "depende_de": ["db", "redis"],
    },
}, ensure_ascii=False)
EJ2_OUT = (
    "  backend:\n"
    "    image: node:18\n"
    "    ports:\n"
    "      - \"4000:4000\"\n"
    "    volumes:\n"
    "      - \".:/app\"\n"
    "    command: npm start\n"
    "    environment:\n"
    "      NODE_ENV: dev\n"
    "      DB_HOST: db\n"
    "    depends_on:\n"
    "      - db\n"
    "      - redis"
)

# Ejemplo 3: sin puerto, con volumen, comando y una sola dependencia
EJ3_IN = json.dumps({
    "servicio": "worker",
    "atributos": {
        "imagen": "python:3.12",
        "volumen": "./src:/srv",
        "comando": "python worker.py",
        "depende_de": ["redis"],
    },
}, ensure_ascii=False)
EJ3_OUT = (
    "  worker:\n"
    "    image: python:3.12\n"
    "    volumes:\n"
    "      - \"./src:/srv\"\n"
    "    command: python worker.py\n"
    "    depends_on:\n"
    "      - redis"
)


def _esperado(datos):
    """Estructura que DEBE tener el servicio en Compose, calculada sin SLM."""
    esp = {}
    if 'imagen' in datos:
        esp['image'] = str(datos['imagen'])
    if 'puerto' in datos:
        esp['ports'] = [f"{datos['puerto']}:{datos['puerto']}"]
    if 'volumen' in datos:
        esp['volumes'] = [str(datos['volumen'])]
    if 'comando' in datos:
        esp['command'] = str(datos['comando'])
    if 'entorno' in datos:
        esp['environment'] = {k: str(v) for k, v in datos['entorno'].items()}
    if 'depende_de' in datos:
        esp['depends_on'] = list(datos['depende_de'])
    return esp


def _validar(nombre, bloque, datos):
    """Devuelve None si el bloque es YAML válido y equivale a lo esperado; si no, un motivo."""
    try:
        doc = yaml.safe_load("services:\n" + bloque)
    except yaml.YAMLError as e:
        return f"YAML inválido: {e}"
    if not isinstance(doc, dict) or not isinstance(doc.get('services'), dict) \
            or list(doc['services'].keys()) != [nombre]:
        return f"debe haber exactamente un servicio llamado '{nombre}' con 2 espacios de sangría"
    real = doc['services'][nombre]
    if not isinstance(real, dict):
        return "el servicio no tiene atributos"
    # Se normaliza a texto para que 5432 y "5432" cuenten igual
    if 'environment' in real and isinstance(real['environment'], dict):
        real['environment'] = {str(k): str(v) for k, v in real['environment'].items()}
    for k in ('image', 'command'):
        if k in real:
            real[k] = str(real[k])
    esp = _esperado(datos)
    if real != esp:
        sobran = sorted(set(real) - set(esp))
        faltan = sorted(set(esp) - set(real))
        distintas = sorted(k for k in esp if k in real and real[k] != esp[k])
        partes = []
        if sobran:
            partes.append(f"sobran las claves {sobran} (no están en el JSON, elimínalas)")
        if faltan:
            partes.append(f"faltan las claves {faltan}")
        if distintas:
            partes.append(f"valores incorrectos en {distintas}")
        return (f"el contenido no coincide: {'; '.join(partes)}. "
                f"Esperado: {json.dumps(esp, ensure_ascii=False)}")
    return None


def _podar(bloque, datos):
    """Quita las claves de nivel servicio que el modelo inventó (no están en la entrada).
    No agrega ni cambia nada: solo elimina lo que sobra."""
    permitidas = set(_esperado(datos))
    salida, conservar = [], True
    for linea in bloque.splitlines():
        sangria = len(linea) - len(linea.lstrip(' '))
        if sangria == 4 and ':' in linea:
            conservar = linea.strip().split(':')[0] in permitidas
        elif sangria <= 2:
            conservar = True
        if conservar:
            salida.append(linea)
    return "\n".join(salida)


def _valor(v):
    """Sin comillas cuando YAML lo lee igual (estilo de los ejemplos); si no, con comillas."""
    t = str(v)
    try:
        if yaml.safe_load("k: " + t) == {"k": t}:
            return t
    except yaml.YAMLError:
        pass
    return json.dumps(t, ensure_ascii=False)


def _bloque_determinista(nombre, datos):
    """Respaldo sin SLM, para cuando el modelo falla o no está disponible.
    Mismo estilo que los ejemplos: ports y volumes entre comillas, el resto solo si hace falta."""
    q = lambda v: json.dumps(str(v), ensure_ascii=False)
    out = [f"  {nombre}:"]
    if 'imagen' in datos:
        out.append(f"    image: {_valor(datos['imagen'])}")
    if 'puerto' in datos:
        puerto = f"{datos['puerto']}:{datos['puerto']}"
        out += ["    ports:", f"      - {q(puerto)}"]
    if 'volumen' in datos:
        out += ["    volumes:", f"      - {q(datos['volumen'])}"]
    if 'comando' in datos:
        out.append(f"    command: {_valor(datos['comando'])}")
    if 'entorno' in datos:
        out.append("    environment:")
        out += [f"      {k}: {_valor(v)}" for k, v in datos['entorno'].items()]
    if 'depende_de' in datos:
        out.append("    depends_on:")
        out += [f"      - {d}" for d in datos['depende_de']]
    return "\n".join(out)


def _limpiar(contenido):
    lineas = []
    for linea in contenido.splitlines():
        s = linea.strip()
        if s.startswith("```") or s.startswith("version:") or s.startswith("services:"):
            continue
        if s:
            lineas.append(linea.rstrip())
    return "\n".join(lineas)


def generar_bloque_servicio(nombre, datos_servicio):
    """
    Recibe la Representación Intermedia (nombre + dict) desde el parser y delega en
    Ollama la síntesis a YAML. La salida del SLM se VALIDA contra lo esperado; si tras
    reintentar no es correcta (o Ollama no responde), se usa un generador determinista.
    """
    payload = json.dumps({"servicio": nombre, "atributos": datos_servicio}, ensure_ascii=False)
    mensajes = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': EJ1_IN},
        {'role': 'assistant', 'content': EJ1_OUT},
        {'role': 'user', 'content': EJ2_IN},
        {'role': 'assistant', 'content': EJ2_OUT},
        {'role': 'user', 'content': EJ3_IN},
        {'role': 'assistant', 'content': EJ3_OUT},
        {'role': 'user', 'content': payload},
    ]

    motivo = None
    for intento in range(1, MAX_INTENTOS + 1):
        try:
            resp = ollama.chat(model=MODELO, messages=mensajes,
                               options={'temperature': 0.0, 'seed': intento})
        except Exception as e:  # Ollama caído, modelo no descargado, etc.
            motivo = f"Ollama no disponible ({e})"
            break

        bloque = _limpiar(resp['message']['content'])
        podado = _podar(bloque, datos_servicio)
        if DEBUG and podado != bloque:
            print(f"[debug] '{nombre}': se eliminaron claves inventadas por el modelo", file=sys.stderr)
        bloque = podado
        motivo = _validar(nombre, bloque, datos_servicio)
        if DEBUG:
            print(f"[debug] '{nombre}' intento {intento}:\n{bloque}\n[debug] -> {motivo or 'OK'}", file=sys.stderr)
        if motivo is None:
            return bloque

        # Reintento: se le muestra al modelo su error
        mensajes += [
            {'role': 'assistant', 'content': bloque},
            {'role': 'user', 'content': f"Incorrecto: {motivo}. Devuelve solo el YAML corregido."},
        ]

    print(f"[aviso] '{nombre}': {motivo}. Se usa el generador de respaldo.", file=sys.stderr)
    return _bloque_determinista(nombre, datos_servicio)

Overwriting traductor/ag_generar_servicio.py


In [10]:
%%file traductor/main.py
import sys
from parser_docker import analizar
from ag_generar_servicio import generar_bloque_servicio


def compilar(entrada):
    """Devuelve (ok, texto). Prioridad de errores: léxico > sintáctico > semántico."""
    r = analizar(entrada)

    if r['lexicos']:
        detalle = "\n".join(f"  Línea {lin}: carácter ilegal {c!r}" for c, lin in r['lexicos'])
        return False, f"Error léxico\n{detalle}"
    if r['sintacticos']:
        detalle = "\n".join(f"  {m}" for m in r['sintacticos'])
        return False, f"Error sintáctico\n{detalle}"
    if r['semanticos']:
        detalle = "\n".join(f"  {m}" for m in r['semanticos'])
        return False, f"Error semántico\n{detalle}"

    # Solo con la entrada ya validada se llama al SLM, un servicio por vez
    lineas = ["version: '3.8'", "services:"]
    for nombre, datos in r['servicios']:
        lineas.append(generar_bloque_servicio(nombre, datos).rstrip())
    return True, "\n".join(lineas)


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Uso: python main.py <archivo.dockermini>")
        sys.exit(1)

    archivo_entrada = sys.argv[1]
    with open(archivo_entrada, 'r', encoding='utf-8') as f:
        entrada = f.read()

    ok, resultado = compilar(entrada)

    # Solo se escribe el .yml si la compilación fue exitosa
    if ok:
        with open(f"{archivo_entrada}.yml", 'w', encoding='utf-8') as out:
            out.write(resultado + '\n')

    print(resultado)
    sys.exit(0 if ok else 1)

Overwriting traductor/main.py


In [11]:
%%file traductor/caso_feliz.dockermini
# Caso feliz: backend completo con base de datos
servicio db {
    imagen: "postgres:15-alpine"
    puerto: 5432
    volumen: "datos_db:/var/lib/postgresql/data"
    entorno: {
        POSTGRES_USER: "admin"
        POSTGRES_PASSWORD: "secretpassword"
    }
}
 
servicio api {
    imagen: "node:18-alpine"
    puerto: 3000
    volumen: ".:/app"
    comando: "npm run start:dev"
    depende_de: [db]
}

Overwriting traductor/caso_feliz.dockermini


In [12]:
%env DOCKERMINI_DEBUG=1
!python traductor/main.py traductor/caso_feliz.dockermini

env: DOCKERMINI_DEBUG=1
version: '3.8'
services:
  db:
    image: postgres:15-alpine
    ports:
      - "5432:5432"
    volumes:
      - "datos_db:/var/lib/postgresql/data"
    environment:
      POSTGRES_USER: admin
      POSTGRES_PASSWORD: secretpassword
  api:
    image: node:18-alpine
    ports:
      - "3000:3000"
    volumes:
      - ".:/app"
    command: npm run start:dev
    depends_on:
      - db


[debug] 'db' intento 1:
  db:
    image: postgres:15-alpine
    ports:
      - "5432:5432"
    volumes:
      - "datos_db:/var/lib/postgresql/data"
    environment:
      POSTGRES_USER: admin
      POSTGRES_PASSWORD: secretpassword
[debug] -> OK
[debug] 'api': se eliminaron claves inventadas por el modelo
[debug] 'api' intento 1:
  api:
    image: node:18-alpine
    ports:
      - "3000:3000"
    volumes:
      - ".:/app"
    command: npm run start:dev
    depends_on:
      - db
[debug] -> OK


In [13]:

!cat traductor/caso_feliz.dockermini.yml
 

version: '3.8'
services:
  db:
    image: postgres:15-alpine
    ports:
      - "5432:5432"
    volumes:
      - "datos_db:/var/lib/postgresql/data"
    environment:
      POSTGRES_USER: admin
      POSTGRES_PASSWORD: secretpassword
  api:
    image: node:18-alpine
    ports:
      - "3000:3000"
    volumes:
      - ".:/app"
    command: npm run start:dev
    depends_on:
      - db


In [14]:

%%file traductor/caso_error_lexico.dockermini
servicio web {
    imagen: "nginx"
    puerto: 80 @
}

Overwriting traductor/caso_error_lexico.dockermini


In [15]:

!python traductor/main.py traductor/caso_error_lexico.dockermini
 

Error léxico
  Línea 3: carácter ilegal '@'


In [16]:
%%file traductor/caso_error_sintax.dockermini
servicio web {
    imagen: "nginx"
    puerto: 80

Writing traductor/caso_error_sintax.dockermini


In [17]:
!python traductor/main.py traductor/caso_error_sintax.dockermini
 

Error sintáctico
  Fin de archivo inesperado (¿falta cerrar una llave o un corchete?)


In [18]:
%%file traductor/caso_error_semantico.dockermini
servicio web {
    puerto: 80
}

Writing traductor/caso_error_semantico.dockermini


In [19]:
!python traductor/main.py traductor/caso_error_semantico.dockermini

Error semántico
  Línea 1: el servicio 'web' no declara 'imagen'


In [20]:
%%file traductor/test.py
import subprocess
import sys
from pathlib import Path

import pytest
import yaml

DIR = Path(__file__).parent
MAIN = DIR / "main.py"


def compilar(tmp_path, fuente):
    """Escribe la fuente, corre main.py como proceso y devuelve (código, stdout, yaml|None)."""
    archivo = tmp_path / "entrada.dockermini"
    archivo.write_text(fuente, encoding="utf-8")
    r = subprocess.run([sys.executable, str(MAIN), str(archivo)],
                       capture_output=True, text=True, cwd=DIR, timeout=300)
    salida = Path(str(archivo) + ".yml")
    doc = yaml.safe_load(salida.read_text(encoding="utf-8")) if salida.exists() else None
    return r.returncode, r.stdout, doc


# ------------------------------------------------------------------ válidos

def test_01_caso_feliz_base(tmp_path):
    fuente = '''
servicio db {
    imagen: "postgres:15-alpine"
    puerto: 5432
}
servicio api {
    imagen: "node:18-alpine"
    puerto: 3000
}
'''
    codigo, _, doc = compilar(tmp_path, fuente)
    assert codigo == 0
    assert doc["version"] == "3.8"
    assert doc["services"]["api"]["ports"] == ["3000:3000"]
    assert doc["services"]["db"]["ports"] == ["5432:5432"]


def test_02_web_simple(tmp_path):
    codigo, _, doc = compilar(tmp_path, 'servicio frontend { imagen: "nginx"\n puerto: 80 }')
    assert codigo == 0
    assert doc["services"]["frontend"]["image"] == "nginx"
    assert doc["services"]["frontend"]["ports"] == ["80:80"]


def test_03_puerto_alto(tmp_path):
    codigo, _, doc = compilar(tmp_path, 'servicio app { imagen: "nginx"\n puerto: 50000 }')
    assert codigo == 0
    assert doc["services"]["app"]["ports"] == ["50000:50000"]


def test_04_tres_servicios(tmp_path):
    fuente = '''
servicio frontend { imagen: "nginx"
 puerto: 80 }
servicio api { imagen: "node:18"
 puerto: 3000 }
servicio db { imagen: "postgres:15"
 puerto: 5432 }
'''
    codigo, _, doc = compilar(tmp_path, fuente)
    assert codigo == 0
    assert list(doc["services"]) == ["frontend", "api", "db"]


def test_05_tags_complejos(tmp_path):
    codigo, _, doc = compilar(tmp_path, 'servicio cache { imagen: "redis:7-alpine" }')
    assert codigo == 0
    assert doc["services"]["cache"]["image"] == "redis:7-alpine"


def test_06_volumen_y_comando(tmp_path):
    fuente = 'servicio api { imagen: "node:18"\n volumen: ".:/app"\n comando: "npm run start:dev" }'
    codigo, _, doc = compilar(tmp_path, fuente)
    assert codigo == 0
    assert doc["services"]["api"]["volumes"] == [".:/app"]
    assert doc["services"]["api"]["command"] == "npm run start:dev"


def test_07_soporte_comentarios(tmp_path):
    fuente = '''# comentario inicial
servicio web {   # comentario al lado
    imagen: "nginx"  # otro
}
'''
    codigo, salida, doc = compilar(tmp_path, fuente)
    assert codigo == 0
    assert "#" not in salida
    assert doc["services"]["web"]["image"] == "nginx"


def test_08_subbloque_entorno(tmp_path):
    fuente = '''servicio db {
    imagen: "postgres:15"
    entorno: {
        POSTGRES_USER: "admin"
        POSTGRES_PASSWORD: "secretpassword"
    }
}'''
    codigo, _, doc = compilar(tmp_path, fuente)
    assert codigo == 0
    assert doc["services"]["db"]["environment"] == {
        "POSTGRES_USER": "admin", "POSTGRES_PASSWORD": "secretpassword"}


def test_09_dependencias_depends_on(tmp_path):
    fuente = '''
servicio db { imagen: "postgres:15" }
servicio redis { imagen: "redis:7-alpine" }
servicio api { imagen: "node:18"
 depende_de: [db, redis] }
'''
    codigo, _, doc = compilar(tmp_path, fuente)
    assert codigo == 0
    assert doc["services"]["api"]["depends_on"] == ["db", "redis"]


def test_10_orden_arbitrario_atributos(tmp_path):
    codigo, _, doc = compilar(tmp_path, 'servicio web { puerto: 80\n imagen: "nginx" }')
    assert codigo == 0
    assert doc["services"]["web"]["image"] == "nginx"
    assert doc["services"]["web"]["ports"] == ["80:80"]


# ------------------------------------------------------------------ errores

@pytest.mark.parametrize("simbolo", ["@", "$", "%"])
def test_11_error_lexico_simbolo_ilegal(tmp_path, simbolo):
    fuente = f'servicio web {{ imagen: "nginx"\n puerto: 80 {simbolo} }}'
    codigo, salida, doc = compilar(tmp_path, fuente)
    assert codigo != 0
    assert "Error léxico" in salida
    assert doc is None


def test_12_error_sintaxis_falta_llave(tmp_path):
    codigo, salida, doc = compilar(tmp_path, 'servicio web { imagen: "nginx"\n puerto: 80\n')
    assert codigo != 0
    assert "Error sintáctico" in salida
    assert doc is None


def test_13_error_sintaxis_declaracion_incompleta(tmp_path):
    codigo, salida, doc = compilar(tmp_path, 'servicio web { imagen:\n puerto: 80 }')
    assert codigo != 0
    assert "Error sintáctico" in salida
    assert doc is None


# ------------------------------------------------- extra: errores semánticos

def test_14_error_semantico_sin_imagen(tmp_path):
    codigo, salida, doc = compilar(tmp_path, 'servicio web { puerto: 80 }')
    assert codigo != 0
    assert "Error semántico" in salida
    assert doc is None


def test_15_error_semantico_dependencia_inexistente(tmp_path):
    codigo, salida, doc = compilar(tmp_path, 'servicio api { imagen: "node"\n depende_de: [fantasma] }')
    assert codigo != 0
    assert "Error semántico" in salida


def test_16_error_semantico_directiva_repetida(tmp_path):
    codigo, salida, _ = compilar(tmp_path, 'servicio web { imagen: "a"\n imagen: "b" }')
    assert codigo != 0
    assert "Error semántico" in salida


def test_17_prioridad_lexico_sobre_sintactico(tmp_path):
    codigo, salida, _ = compilar(tmp_path, 'servicio web { imagen: @\n puerto: }')
    assert "Error léxico" in salida
    assert "Error sintáctico" not in salida

Overwriting traductor/test.py


In [21]:
!python -m pytest -v traductor/test.py

============================= test session starts =============================
platform win32 -- Python 3.12.3, pytest-9.1.1, pluggy-1.7.0 -- C:\Users\lucas\taller-lenguajes\tp_traductor\venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\lucas\taller-lenguajes\tp_traductor
plugins: anyio-4.15.1, platformdirs-4.13.5
collecting ... collected 19 items

traductor/test.py::test_01_caso_feliz_base PASSED                        [  5%]
traductor/test.py::test_02_web_simple PASSED                             [ 10%]
traductor/test.py::test_03_puerto_alto PASSED                            [ 15%]
traductor/test.py::test_04_tres_servicios PASSED                         [ 21%]
traductor/test.py::test_05_tags_complejos PASSED                         [ 26%]
traductor/test.py::test_06_volumen_y_comando PASSED                      [ 31%]
traductor/test.py::test_07_soporte_comentarios PASSED                    [ 36%]
traductor/test.py::test_08_subbloque_entorno PASSED                     